# DentoSeg-SSL — Unified Run Notebook
Self-Supervised Dental Panoramic Segmentation · Kaggle / Colab / Local

## Cell 1 · Environment Detection & Dependency Install

In [ ]:
import os, sys, subprocess, pathlib, importlib

def detect_env():
    if os.path.exists('/kaggle'):
        return 'kaggle'
    if 'COLAB_GPU' in os.environ or 'COLAB_BACKEND_URL' in os.environ:
        return 'colab'
    return 'local'

ENV = detect_env()
print(f'Environment: {ENV}')

if ENV in ('kaggle', 'colab'):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', 'natsort', 'pyyaml'],
        check=True
    )
    print('Packages installed.')
else:
    print('Local — ensure: pip install -r requirements.txt')

## Cell 2 · Clone / Pull Repo & Wire sys.path

In [ ]:
REPO_URL  = 'https://github.com/MNADITYA05/DentoSeg-SSL-Self-Supervised-Dental-Panoramic-Segmentation.git'
REPO_NAME = 'DentoSeg-SSL-Self-Supervised-Dental-Panoramic-Segmentation'

if ENV == 'kaggle':
    REPO_ROOT = pathlib.Path('/kaggle/working') / REPO_NAME
elif ENV == 'colab':
    REPO_ROOT = pathlib.Path('/content') / REPO_NAME
else:
    REPO_ROOT = pathlib.Path('..').resolve()

if ENV in ('kaggle', 'colab'):
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_ROOT)], check=True)
        print(f'Repo cloned → {REPO_ROOT}')
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--quiet'], check=True)
        print(f'Repo updated → {REPO_ROOT}')

# Wire path and invalidate Python module cache so fresh code is picked up
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

importlib.invalidate_caches()
print(f'sys.path wired: {REPO_ROOT}')

## Cell 3 · Imports

In [ ]:
import tensorflow as tf
import numpy as np

from data.dataset        import DentalDataset
from data.augmentation   import RandomContrast
from models.encoder      import create_encoder
from models.segmentation import create_contrastive_model, create_segmentation_model
from training.pretrain   import ContrastiveTrainer
from training.finetune   import train_segmentation
from training.losses     import dice_loss
from evaluation.metrics  import iou_metric, dice_coefficient
from evaluation.visualize import visualize_results, plot_training_history
from utils.seed          import set_global_seed

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## Cell 4 · Configuration

In [ ]:
SEED = 42
set_global_seed(SEED)

# ── Dataset paths: auto-detect correct Kaggle mount ──────────────────
# Kaggle mounts datasets from other users under /kaggle/input/datasets/<username>/
# This checks both possible locations so it works regardless of how the dataset was added.
_KAGGLE_CANDIDATES = [
    '/kaggle/input/datasets/truthisneverlinear/childrens-dental-panoramic-radiographs-dataset',
    '/kaggle/input/childrens-dental-panoramic-radiographs-dataset',
]
_SEG_SUBPATH = 'Dental_dataset/Adult tooth segmentation dataset/Panoramic radiography database'

if ENV == 'kaggle':
    DATASET_BASE = None
    for candidate in _KAGGLE_CANDIDATES:
        if pathlib.Path(candidate).exists():
            DATASET_BASE = candidate
            break
    if DATASET_BASE is None:
        raise FileNotFoundError(
            'Dataset not found. Make sure you have added the '
            'Children\'s Dental Panoramic Radiographs dataset to this notebook.'
        )
    IMAGES_PATH = f'{DATASET_BASE}/{_SEG_SUBPATH}/images'
    MASKS_PATH  = f'{DATASET_BASE}/{_SEG_SUBPATH}/mask'
    OUTPUT_DIR  = '/kaggle/working/outputs'

elif ENV == 'colab':
    IMAGES_PATH = '/content/data/images'
    MASKS_PATH  = '/content/data/masks'
    OUTPUT_DIR  = '/content/outputs'

else:
    IMAGES_PATH = 'data/images'
    MASKS_PATH  = 'data/masks'
    OUTPUT_DIR  = 'outputs'

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Images : {IMAGES_PATH}')
print(f'Masks  : {MASKS_PATH}')
print(f'Outputs: {OUTPUT_DIR}')

# Verify paths exist before proceeding
assert pathlib.Path(IMAGES_PATH).exists(), f'Images path not found: {IMAGES_PATH}'
assert pathlib.Path(MASKS_PATH).exists(),  f'Masks path not found: {MASKS_PATH}'
print('Paths verified OK.')

# ── Hyperparameters ───────────────────────────────────────────────────
IMAGE_SIZE   = 224
CHANNELS     = 1
INPUT_SHAPE  = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)

BATCH_SIZE_PRETRAIN   = 16
EPOCHS_PRETRAIN       = 50
LR_PRETRAIN           = 3e-4
TEMPERATURE           = 0.1
PROJECTION_DIM        = 128

BATCH_SIZE_FINETUNE   = 16
EPOCHS_FINETUNE       = 50
LR_FINETUNE           = 1e-4
FREEZE_ENCODER_EPOCHS = 10

AUG_CFG = dict(
    horizontal_flip       = True,
    rotation_factor       = 0.1,
    gaussian_noise_stddev = 0.05,
    contrast_lower        = 0.85,
    contrast_upper        = 1.15,
)

ENCODER_CKPT = f'{OUTPUT_DIR}/pretrain/encoder_pretrained.keras'
BEST_MODEL   = f'{OUTPUT_DIR}/finetune/best_model.keras'
FINAL_MODEL  = f'{OUTPUT_DIR}/finetune/dental_segmentation_model.keras'
RESULTS_PLOT = f'{OUTPUT_DIR}/finetune/segmentation_results.png'
HISTORY_PLOT = f'{OUTPUT_DIR}/finetune/training_history.png'

print('Configuration ready.')

## Cell 5 · Load Data

In [ ]:
dental_dataset = DentalDataset(IMAGES_PATH, MASKS_PATH, image_size=IMAGE_SIZE)
(train_images, train_masks), (test_images, test_masks) = dental_dataset.prepare_data(
    test_ratio=0.2,
    seed=SEED,
)
print(f'Train : {train_images.shape}')
print(f'Test  : {test_images.shape}')

## Cell 6 · Build Contrastive Model

In [ ]:
encoder = create_encoder(input_shape=INPUT_SHAPE)
contrastive_model = create_contrastive_model(
    encoder          = encoder,
    input_shape      = INPUT_SHAPE,
    projection_dim   = PROJECTION_DIM,
    augmentation_cfg = AUG_CFG,
)
contrastive_model.summary()

## Cell 7 · Contrastive SSL Pretraining

In [ ]:
pretrain_ds = (
    tf.data.Dataset.from_tensor_slices(train_images)
    .shuffle(len(train_images), seed=SEED)
    .batch(BATCH_SIZE_PRETRAIN)
    .prefetch(tf.data.AUTOTUNE)
)

trainer = ContrastiveTrainer(
    model         = contrastive_model,
    encoder       = encoder,
    temperature   = TEMPERATURE,
    learning_rate = LR_PRETRAIN,
    log_path      = f'{OUTPUT_DIR}/pretrain/pretrain_log.csv',
)

trainer.train(
    dataset            = pretrain_ds,
    epochs             = EPOCHS_PRETRAIN,
    encoder_checkpoint = ENCODER_CKPT,
)

## Cell 8 · Build Segmentation Model
Loads pretrained encoder if checkpoint exists, otherwise uses the encoder from Cell 6.

In [ ]:
if pathlib.Path(ENCODER_CKPT).exists():
    print(f'Loading pretrained encoder from {ENCODER_CKPT} …')
    encoder = tf.keras.models.load_model(
        ENCODER_CKPT,
        custom_objects={'RandomContrast': RandomContrast},
    )
else:
    print('No checkpoint found — using encoder from Cell 6.')

seg_model = create_segmentation_model(encoder=encoder, input_shape=INPUT_SHAPE)
seg_model.summary()

## Cell 9 · Segmentation Fine-tuning

In [ ]:
history = train_segmentation(
    model                 = seg_model,
    encoder               = encoder,
    train_images          = train_images,
    train_masks           = train_masks,
    val_ratio             = 0.2,
    batch_size            = BATCH_SIZE_FINETUNE,
    epochs                = EPOCHS_FINETUNE,
    learning_rate         = LR_FINETUNE,
    freeze_encoder_epochs = FREEZE_ENCODER_EPOCHS,
    best_model_path       = BEST_MODEL,
    final_model_path      = FINAL_MODEL,
    log_csv_path          = f'{OUTPUT_DIR}/finetune/finetune_log.csv',
    seed                  = SEED,
)

## Cell 10 · Evaluation

In [ ]:
seg_model.compile(
    optimizer = tf.keras.optimizers.Adam(),
    loss      = dice_loss,
    metrics   = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        iou_metric,
        dice_coefficient,
    ],
)

results = seg_model.evaluate(test_images, test_masks, verbose=1)
metric_names = ['Loss', 'Accuracy', 'Precision', 'Recall', 'IoU', 'Dice']
print('\n── Test Results ──────────────────')
for name, val in zip(metric_names, results):
    print(f'  {name:<12}: {val:.4f}')

## Cell 11 · Visualisation

In [ ]:
plot_training_history(history, save_path=HISTORY_PLOT)

In [ ]:
visualize_results(
    seg_model, test_images, test_masks,
    num_samples = 5,
    save_path   = RESULTS_PLOT,
    seed        = SEED,
)

## Cell 12 · Load Saved Model (Inference Only)
Skip all training cells and run only this cell to load a previously saved model.

In [ ]:
MODEL_TO_LOAD = FINAL_MODEL  # or BEST_MODEL

loaded_model = tf.keras.models.load_model(
    MODEL_TO_LOAD,
    custom_objects={
        'RandomContrast'  : RandomContrast,
        'dice_loss'       : dice_loss,
        'iou_metric'      : iou_metric,
        'dice_coefficient': dice_coefficient,
    },
)
print(f'Model loaded from {MODEL_TO_LOAD}')
loaded_model.summary()